# CFPB Fraud Intelligence — End-to-End Colab Demo

This notebook clones your GitHub repository, installs the minimal runtime dependencies, and
runs the required single-command pipeline:

```bash
python src/model_runner.py
```

Before running, set the repository URL and branch in the next cell. For a private repository,
create a Colab secret named `GITHUB_TOKEN` with read access to that repository.


In [1]:
REPO_URL = "https://github.com/michaelbimo/Milestone-4-Model-Pipeline-Implementation.git"
BRANCH = "main"
PROJECT_FOLDER = "Milestone-4-Model-Pipeline-Implementation"


In [2]:
from pathlib import Path
import os
import subprocess

project_dir = Path("/content") / PROJECT_FOLDER

# Optional private-repository authentication through Colab Secrets.
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

env = os.environ.copy()
if token:
    env["GITHUB_TOKEN"] = token
    askpass = Path("/tmp/git-askpass.sh")
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*Password*) echo "$GITHUB_TOKEN" ;;\nesac\n'
    )
    askpass.chmod(0o700)
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin"], check=True, env=env)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True, env=env)
    subprocess.run(["git", "-C", str(project_dir), "pull", "origin", BRANCH], check=True, env=env)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)],
        check=True,
        env=env,
    )

os.chdir(project_dir)
print("Working directory:", Path.cwd())


Working directory: /content/Milestone-4-Model-Pipeline-Implementation


In [3]:
# Use the latest Colab runtime and install project-specific packages.
%pip install -q -r requirements.txt


In [4]:
import torch
from pathlib import Path

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_path = Path("data/processed/complaints_clean.csv.gz")
print("Preprocessed dataset exists:", data_path.exists())
if data_path.exists():
    print("Dataset size (MB):", round(data_path.stat().st_size / 1_000_000, 2))


GPU available: True
GPU: Tesla T4
Preprocessed dataset exists: True
Dataset size (MB): 15.55


## Optional fast validation

This checks loading, sample selection, prompt construction, and output paths without
downloading FLAN-T5.


In [5]:
!python src/model_runner.py --prepare-only

Loaded 25,576 preprocessed complaints.
Selected 10 representative samples.
Prepared prompts saved to: /content/Milestone-4-Model-Pipeline-Implementation/outputs/prepared_inputs.jsonl


## Required full run

Use a GPU runtime (`Runtime` → `Change runtime type` → `T4 GPU`) and run the command below.
The first run downloads and caches `google/flan-t5-small`.


In [6]:
!python src/model_runner.py

config.json: 100% 1.40k/1.40k [00:00<00:00, 5.18MB/s]
tokenizer_config.json: 100% 2.54k/2.54k [00:00<00:00, 11.7MB/s]

spiece.model: downloading bytes:  36% 281k/792k [00:00<00:01, 413kB/s, 7.12kB/s  ]
spiece.model: downloading bytes: 100% 649k/649k [00:00<00:00, 750kB/s, 63.6kB/s  ]
spiece.model: reconstructing file: 100% 792k/792k [00:00<00:00, 915kB/s, 77.9kB/s  ]
tokenizer.json: 100% 2.42M/2.42M [00:00<00:00, 83.2MB/s]
special_tokens_map.json: 100% 2.20k/2.20k [00:00<00:00, 9.08MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:  71% 218M/308M [00:01<00:00, 274MB/s, 17.5MB/s  ]
model.safetensors: downloading bytes:  87% 266M/308M [00:01<00:00, 259MB/s, 22.5MB/s  ]
model.safetensors: downloading bytes: 100% 291M/291M [00:02<00:00, 145MB/s, 27.1MB/s  ]
model.safetensors: reconstructing file: 100% 308M/308M [00:02<00:00, 153MB/s, 29.0MB/s  ]
Loading weights: 100% 190/190 [00:00<00:00, 1170.30it/s]
[transformers] The tied weight

In [7]:
from pathlib import Path
from IPython.display import Markdown, display

samples_path = Path("outputs/samples.txt")
metadata_path = Path("outputs/run_metadata.json")

print(samples_path.read_text(encoding="utf-8"))
display(Markdown(Path("outputs/output_description.md").read_text(encoding="utf-8")))
print(metadata_path.read_text(encoding="utf-8"))


CFPB GENERATIVE FRAUD INTELLIGENCE — PRELIMINARY SAMPLES

SAMPLE 1
Complaint ID: 11817563
Product / Sub-product: Money transfer, virtual currency, or money service / Mobile or digital wallet
Provisional archetype: money-transfer or payment-app scam
Red flags: urgent or threatening language; request to transfer money
Risk: Medium (score 3)
Generated summary:
i tried to call redacted times they hung up in mid ring. i tried to unlink my cards from my bank accounts and you can't even click on it continues to blink and clicking on it.
--------------------------------------------------------------------

SAMPLE 2
Complaint ID: 13817257
Product / Sub-product: Money transfer, virtual currency, or money service / Domestic (US) money transfer
Provisional archetype: debt-collection or government impostor scam
Red flags: impersonation of a trusted organization
Risk: Low (score 1)
Generated summary:
A redacted redacted , married woman on redacted living in redacted az. i answered the redacted text 

# Preliminary output description

The pipeline selected **10** representative CFPB complaint narratives and used **google/flan-t5-small** with deterministic beam-search decoding. Each prompt included the cleaned complaint, a provisional transparent archetype, extracted red flags, and a rule-based risk level. The outputs are preliminary and require human review because the generator is not an independent fact checker and the provisional archetypes are not the final trained classifier labels.

Run mode: **full generation**.


{
  "started_at_utc": "2026-07-26T18:36:40.224891+00:00",
  "finished_at_utc": "2026-07-26T18:37:02.000778+00:00",
  "dataset_path": "data/processed/complaints_clean.csv.gz",
  "dataset_rows_loaded": 25576,
  "samples_selected": 10,
  "model_source": "google/flan-t5-small",
  "device": "cuda",
  "prepare_only": false,
  "quality_checks": {
    "non_empty_output_rate": 1.0,
    "average_summary_words": 21.7,
    "risk_level_mention_rate": 0.0
  }
}



In [8]:
from pathlib import Path
import os

PROJECT_DIR = Path(
    "/content/Milestone-4-Model-Pipeline-Implementation"
)

os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
print("\nGenerated outputs:")

for path in sorted(Path("outputs").glob("*")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

Project directory: /content/Milestone-4-Model-Pipeline-Implementation

Generated outputs:
README.md: 638 bytes
output_description.md: 528 bytes
prepared_inputs.jsonl: 26,611 bytes
run_metadata.json: 452 bytes
samples.jsonl: 28,143 bytes
samples.txt: 5,171 bytes


## Before closing Colab

Download or commit the following small files:

- `outputs/samples.txt`
- `outputs/samples.jsonl`
- `outputs/run_metadata.json`
- `outputs/output_description.md`

Do not commit model caches or checkpoints.
